In [1]:
# Cell 1: Setup & Constants
# Notebook 03: Fact_Transaction — Gold_SalesOps_Fact_Transaction
# Source: TransactionDetail (Silver) + Transaction (ClientPartyId) + Policy (InceptionDate) + Product (product fallback)
# Three LEFT JOINs. PolicyId and ProductId are unique in their tables.
# Set RUN_STATS = False for fast production runs, True for debugging with full stats.

from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

SILVER_BASE = "abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo"
RUN_STATS = False

print("Setup complete.")
print(f"Silver base: {SILVER_BASE}")
print(f"RUN_STATS: {RUN_STATS}")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 3, Finished, Available, Finished, False)

Setup complete.
Silver base: abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo
RUN_STATS: False


In [2]:
# Cell 2: Load TransactionDetail

df_td = (
    spark.read.format("delta").load(f"{SILVER_BASE}/TransactionDetail")
    .filter(F.col("IsDeleted") == False)
    .filter(F.col("PolicyId") != -1)
    .filter(F.col("PolicyId").isNotNull())
)

if RUN_STATS:
    td_count = df_td.count()
    print(f"TransactionDetail rows: {td_count:,}\n")

    print("Key column NULL counts:")
    print("-" * 55)
    for col_name in ["PolicyId", "PolicySectionId", "TransactionId", "GlobalPartyId", "GlobalProductLineId", "ProductId", "GlobalFinancialGeographyId", "GlobalFinancialSegmentId", "USDExchangeRate"]:
        null_count = df_td.filter(F.col(col_name).isNull()).count()
        pct = null_count / td_count * 100 if td_count > 0 else 0
        print(f"  {col_name:<35} {null_count:>12,}  ({pct:5.1f}%)")

    display(df_td.limit(5))
else:
    print("TransactionDetail loaded.")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 4, Finished, Available, Finished, False)

TransactionDetail loaded.


In [3]:
# Cell 3: Load Transaction table (for InsuredPartyId / ClientPartyId)

df_txn = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Transaction")
    .filter(F.col("IsDeleted") == False)
    .select("TransactionId", "InsuredPartyId", "ClientPartyId")
)

if RUN_STATS:
    txn_count = df_txn.count()
    txn_distinct = df_txn.select("TransactionId").distinct().count()

    print(f"Transaction rows:          {txn_count:,}")
    print(f"Distinct TransactionId:    {txn_distinct:,}")
    print(f"TransactionId unique?      {'YES' if txn_distinct == txn_count else 'NO — DUPLICATES EXIST, DEDUP NEEDED'}")

    null_insured = df_txn.filter(F.col("InsuredPartyId").isNull()).count()
    null_client = df_txn.filter(F.col("ClientPartyId").isNull()).count()
    print(f"\nInsuredPartyId NULL:       {null_insured:,} ({null_insured/txn_count*100:.1f}%)")
    print(f"ClientPartyId NULL:        {null_client:,} ({null_client/txn_count*100:.1f}%)")

    display(df_txn.limit(5))
else:
    print("Transaction loaded.")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 5, Finished, Available, Finished, False)

Transaction loaded.


In [4]:
# Cell 4: Join TD + Transaction on TransactionId → get ClientPartyId

before_count = df_td.count()

df_joined = df_td.join(df_txn, on="TransactionId", how="left")

after_count = df_joined.count()

# DUPLICATE CHECK — always runs
print(f"Row count BEFORE Transaction join: {before_count:,}")
print(f"Row count AFTER Transaction join:  {after_count:,}")
print(f"STATUS: {'No duplicates' if before_count == after_count else f'DUPLICATES DETECTED ({after_count - before_count:,} extra rows)'}")

# COALESCE: InsuredPartyId first, ClientPartyId as fallback
df_joined = df_joined.withColumn(
    "ClientPartyId",
    F.coalesce(F.col("InsuredPartyId"), F.col("ClientPartyId"))
).drop("InsuredPartyId")

if RUN_STATS:
    used = df_joined.filter(F.col("ClientPartyId").isNotNull()).count()
    still_null = after_count - used
    print(f"\nClientPartyId populated:  {used:,} ({used/after_count*100:.1f}%)")
    print(f"ClientPartyId NULL:       {still_null:,} ({still_null/after_count*100:.1f}%)")
    display(df_joined.select("TransactionId", "ClientPartyId", "GlobalPartyId").limit(10))

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 6, Finished, Available, Finished, False)

Row count BEFORE Transaction join: 426,082,942
Row count AFTER Transaction join:  426,082,942
STATUS: No duplicates


In [5]:
# Cell 5: Load Policy

df_policy = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Policy")
    .filter(F.col("IsDeleted") == False)
    .select("PolicyId", "InceptionDate", "FirstInceptionDate", "ExpiryDate", "RenewalDate", "RefInsuranceTypeId", "RefInsuranceType")
)

if RUN_STATS:
    policy_count = df_policy.count()
    policy_distinct = df_policy.select("PolicyId").distinct().count()

    print(f"Policy rows:         {policy_count:,}")
    print(f"Distinct PolicyId:   {policy_distinct:,}")
    print(f"PolicyId unique?     {'YES' if policy_distinct == policy_count else 'NO — DUPLICATES EXIST'}")

    display(df_policy.limit(5))
else:
    print("Policy loaded.")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 7, Finished, Available, Finished, False)

Policy loaded.


In [6]:
# Cell 6: Load Product

df_product = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Product")
    .filter(F.col("IsDeleted") == False)
    .select(
        F.col("ProductId"),
        F.col("GlobalProductClassId").alias("_p_GlobalProductClassId"),
        F.col("GlobalProductClass").alias("_p_GlobalProductClass"),
        F.col("GlobalProductLineId").alias("_p_GlobalProductLineId"),
        F.col("GlobalProductLine").alias("_p_GlobalProductLine"),
        F.col("GlobalProductId").alias("_p_GlobalProductId"),
        F.col("GlobalProduct").alias("_p_GlobalProduct"),
    )
)

if RUN_STATS:
    product_count = df_product.count()
    product_distinct = df_product.select("ProductId").distinct().count()
    has_global = df_product.filter(F.col("_p_GlobalProductLineId").isNotNull()).count()

    print(f"Product rows:            {product_count:,}")
    print(f"ProductId unique?        {'YES' if product_distinct == product_count else 'NO'}")
    print(f"Has GlobalProductLineId: {has_global:,} ({has_global/product_count*100:.1f}%)")

    display(df_product.limit(5))
else:
    print("Product loaded.")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 8, Finished, Available, Finished, False)

Product loaded.


In [7]:
# Cell 7: Join + Policy on PolicyId

before_count = df_joined.count()

df_joined = df_joined.join(df_policy, on="PolicyId", how="left")

after_count = df_joined.count()

# DUPLICATE CHECK — always runs
print(f"Row count BEFORE Policy join: {before_count:,}")
print(f"Row count AFTER Policy join:  {after_count:,}")
print(f"STATUS: {'No duplicates' if before_count == after_count else f'DUPLICATES DETECTED ({after_count - before_count:,} extra rows)'}")

if RUN_STATS:
    matched = df_joined.filter(F.col("InceptionDate").isNotNull()).count()
    unmatched = after_count - matched
    print(f"\nInceptionDate matched:   {matched:,} ({matched/after_count*100:.1f}%)")
    print(f"InceptionDate NULL:      {unmatched:,} ({unmatched/after_count*100:.1f}%)")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 9, Finished, Available, Finished, False)

Row count BEFORE Policy join: 426,082,942
Row count AFTER Policy join:  426,082,942
STATUS: No duplicates


In [8]:
# Cell 8: Join + Product on ProductId (product fallback)

before_count = df_joined.count()

df_joined = df_joined.join(df_product, on="ProductId", how="left")

after_count = df_joined.count()

# COALESCE: TD product first, Product table fallback
df_joined = df_joined.withColumn(
    "GlobalProductClassId", F.coalesce(F.col("GlobalProductClassId"), F.col("_p_GlobalProductClassId"))
).withColumn(
    "GlobalProductClass", F.coalesce(F.col("GlobalProductClass"), F.col("_p_GlobalProductClass"))
).withColumn(
    "GlobalProductLineId", F.coalesce(F.col("GlobalProductLineId"), F.col("_p_GlobalProductLineId"))
).withColumn(
    "GlobalProductLine", F.coalesce(F.col("GlobalProductLine"), F.col("_p_GlobalProductLine"))
).withColumn(
    "GlobalProductId", F.coalesce(F.col("GlobalProductId"), F.col("_p_GlobalProductId"))
).withColumn(
    "GlobalProduct", F.coalesce(F.col("GlobalProduct"), F.col("_p_GlobalProduct"))
).drop(
    "_p_GlobalProductClassId", "_p_GlobalProductClass",
    "_p_GlobalProductLineId", "_p_GlobalProductLine",
    "_p_GlobalProductId", "_p_GlobalProduct",
)

# DUPLICATE CHECK — always runs
print(f"Row count BEFORE Product join: {before_count:,}")
print(f"Row count AFTER Product join:  {after_count:,}")
print(f"STATUS: {'No duplicates' if before_count == after_count else f'DUPLICATES DETECTED ({after_count - before_count:,} extra rows)'}")

if RUN_STATS:
    total = after_count
    has_product = df_joined.filter(F.col("GlobalProductLineId").isNotNull()).count()
    still_null = total - has_product
    print(f"\nProduct coverage after COALESCE:")
    print(f"  Has GlobalProductLineId: {has_product:,} ({has_product/total*100:.1f}%)")
    print(f"  Still NULL:              {still_null:,} ({still_null/total*100:.1f}%)")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 10, Finished, Available, Finished, False)

Row count BEFORE Product join: 426,082,942
Row count AFTER Product join:  426,082,942
STATUS: No duplicates


In [9]:
# Cell 8b: Load Organisation + Join on OwnershipOrganisationId

df_org = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Organisation")
    .filter(F.col("IsDeleted") == False)
    .select(
        F.col("OrganisationId"),
        F.col("Organisation"),
    )
)

before_count = df_joined.count()

df_joined = df_joined.join(df_org, df_joined["OwnershipOrganisationId"] == df_org["OrganisationId"], how="left").drop("OrganisationId")

after_count = df_joined.count()

# DUPLICATE CHECK — always runs
print(f"Row count BEFORE Organisation join: {before_count:,}")
print(f"Row count AFTER Organisation join:  {after_count:,}")
print(f"STATUS: {'No duplicates' if before_count == after_count else f'DUPLICATES DETECTED ({after_count - before_count:,} extra rows)'}")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 11, Finished, Available, Finished, False)

Row count BEFORE Organisation join: 426,082,942
Row count AFTER Organisation join:  426,082,942
STATUS: No duplicates


In [10]:
# Cell 9: USD Conversion + TotalWTWRevenueUSD

revenue_cols = ["GrossBrokerage", "NetBrokerage", "GrossFee", "NetFee", "AdditionalCommission", "ContingentCommission", "MarketDerivedIncome", "GrossPremium", "NetPremium"]

for col_name in revenue_cols:
    df_joined = df_joined.withColumn(
        f"{col_name}USD",
        F.round(F.coalesce(F.col(col_name), F.lit(0)) * F.coalesce(F.col("USDExchangeRate"), F.lit(0)), 2)
    )

df_joined = df_joined.withColumn(
    "TotalWTWRevenueUSD",
    F.round(
        F.col("NetBrokerageUSD")
        + F.col("NetFeeUSD")
        + F.col("AdditionalCommissionUSD")
        + F.col("ContingentCommissionUSD")
        + F.col("MarketDerivedIncomeUSD"),
        2,
    )
)

# Adjusted brokerage: net if available and non-zero, else gross
df_joined = df_joined.withColumn(
    "NetBrokerageUSD_Adj",
    F.when(
        (F.col("NetBrokerageUSD").isNull()) | (F.col("NetBrokerageUSD") == 0),
        F.col("GrossBrokerageUSD")
    ).otherwise(F.col("NetBrokerageUSD"))
)

# Adjusted fee: net if available and non-zero, else gross
df_joined = df_joined.withColumn(
    "NetFeeUSD_Adj",
    F.when(
        (F.col("NetFeeUSD").isNull()) | (F.col("NetFeeUSD") == 0),
        F.col("GrossFeeUSD")
    ).otherwise(F.col("NetFeeUSD"))
)

# Adjusted total WTW revenue using adjusted brokerage and fee
df_joined = df_joined.withColumn(
    "TotalWTWRevenueUSD_Adj",
    F.round(
        F.col("NetBrokerageUSD_Adj")
        + F.col("NetFeeUSD_Adj")
        + F.col("AdditionalCommissionUSD")
        + F.col("ContingentCommissionUSD")
        + F.col("MarketDerivedIncomeUSD"),
        2,
    )
)

# UWPartyId: for ShareBroker rows, extract UW from Party1-4; for UW/Carrier rows, use AccountPartyId
df_joined = df_joined.withColumn(
    "UWPartyId",
    F.when(F.col("AccountPartyRole") == "ShareBroker",
        F.coalesce(
            F.when(F.col("Party1Role") == "UW", F.col("Party1Id")),
            F.when(F.col("Party2Role") == "UW", F.col("Party2Id")),
            F.when(F.col("Party3Role") == "UW", F.col("Party3Id")),
            F.when(F.col("Party4Role") == "UW", F.col("Party4Id"))
        )
    )
    .when(F.col("AccountPartyRole").isin("UW", "Carrier"), F.col("AccountPartyId"))
    .otherwise(None)
)

# PoolPartyId: for ShareBroker rows, extract Pool from Party1-4; for Pool rows, use AccountPartyId
df_joined = df_joined.withColumn(
    "PoolPartyId",
    F.when(F.col("AccountPartyRole") == "ShareBroker",
        F.coalesce(
            F.when(F.col("Party1Role") == "Pool", F.col("Party1Id")),
            F.when(F.col("Party2Role") == "Pool", F.col("Party2Id")),
            F.when(F.col("Party3Role") == "Pool", F.col("Party3Id")),
            F.when(F.col("Party4Role") == "Pool", F.col("Party4Id"))
        )
    )
    .when(F.col("AccountPartyRole") == "Pool", F.col("AccountPartyId"))
    .otherwise(None)
)

# Derived year
df_joined = df_joined.withColumn("InceptionYear", F.year(F.col("InceptionDate")))

if RUN_STATS:
    print("USD conversion complete. Spot-check sample:")
    display(
        df_joined
        .filter(F.col("NetBrokerage").isNotNull())
        .filter(F.col("NetBrokerage") != 0)
        .select("GlobalCurrencyCode", "USDExchangeRate", "NetBrokerage", "NetBrokerageUSD", "TotalWTWRevenueUSD")
        .limit(10)
    )
else:
    print("USD conversion complete.")


StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 12, Finished, Available, Finished, False)

USD conversion complete.


In [11]:
# Cell 10: Select Final Columns

fact_transaction = df_joined.select(
    # IDs & Keys
    "TransactionDetailId", "PolicyId", "PolicySectionId", "TransactionId", "SourceId",
    # Client (from Transaction table join)
    "ClientPartyId",
    # Parties (original from TransactionDetail)
    "GlobalPartyId", "GlobalPartyRoleId", "AccountPartyId", "AccountPartyRole",
    # Derived UW/Pool (from Party1-4 for ShareBroker rows)
    "UWPartyId", "PoolPartyId",
    # Organisation (WTW team — from Organisation table)
    "Organisation",
    # Dimensions
    "GlobalFinancialGeographyId", "GlobalFinancialSegmentId",
    # Dates
    "TransactionDate", "TransactionDetailDate", "GLAccountingDate", "InceptionDate", "FirstInceptionDate", "ExpiryDate", "RenewalDate", "InceptionYear",
    # Policy
    "RefInsuranceTypeId", "RefInsuranceType",
    # Product
    "ProductKey",
    "GlobalProductClassId", "GlobalProductClass",
    "GlobalProductLineId", "GlobalProductLine",
    "GlobalProductId", "GlobalProduct",
    # Currency
    "GlobalCurrencyCode", "USDExchangeRate",
    # Revenue (local)
    "GrossBrokerage", "NetBrokerage", "GrossFee", "NetFee", "AdditionalCommission", "ContingentCommission",
    "MarketDerivedIncome", "GrossPremium", "NetPremium",
    # Revenue (USD)
    "GrossBrokerageUSD", "NetBrokerageUSD", "GrossFeeUSD", "NetFeeUSD", "AdditionalCommissionUSD", "ContingentCommissionUSD",
    "MarketDerivedIncomeUSD", "GrossPremiumUSD", "NetPremiumUSD", "TotalWTWRevenueUSD",
    "NetBrokerageUSD_Adj", "NetFeeUSD_Adj", "TotalWTWRevenueUSD_Adj",
)

if RUN_STATS:
    row_count = fact_transaction.count()
    print(f"Fact_Transaction rows: {row_count:,}\n")

    print("Column NULL counts:")
    print("-" * 60)
    for col_name in fact_transaction.columns:
        null_count = fact_transaction.filter(F.col(col_name).isNull()).count()
        pct = null_count / row_count * 100 if row_count > 0 else 0
        print(f"  {col_name:<35} {null_count:>12,}  ({pct:5.1f}%)")
else:
    print("Final columns selected.")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 13, Finished, Available, Finished, False)

Final columns selected.


In [12]:
# Cell 11: Data Quality Overview (only runs when RUN_STATS = True)

if RUN_STATS:
    row_count = fact_transaction.count()

    print("=== AccountPartyRole Distribution ===")
    display(
        fact_transaction.groupBy("AccountPartyRole")
        .agg(F.count("*").alias("Count"), F.round(F.count("*") / row_count * 100, 1).alias("Pct"))
        .orderBy("Count", ascending=False)
    )

    print("\n=== RefInsuranceType Distribution ===")
    display(
        fact_transaction.groupBy("RefInsuranceType")
        .agg(F.count("*").alias("Count"), F.round(F.count("*") / row_count * 100, 1).alias("Pct"))
        .orderBy("Count", ascending=False)
    )

    print("\n=== Top 10 GlobalProductLine ===")
    display(
        fact_transaction.groupBy("GlobalProductLine")
        .agg(F.count("*").alias("Count"), F.round(F.count("*") / row_count * 100, 1).alias("Pct"))
        .orderBy("Count", ascending=False)
        .limit(10)
    )

    print("\n=== Top 10 Currencies ===")
    display(
        fact_transaction.groupBy("GlobalCurrencyCode")
        .agg(F.count("*").alias("Count"), F.round(F.count("*") / row_count * 100, 1).alias("Pct"))
        .orderBy("Count", ascending=False)
        .limit(10)
    )

    print("\n=== InceptionYear Distribution ===")
    display(
        fact_transaction.groupBy("InceptionYear")
        .agg(F.count("*").alias("Count"))
        .orderBy("InceptionYear", ascending=False)
        .limit(20)
    )

    print("\n=== TotalWTWRevenueUSD Summary ===")
    display(
        fact_transaction.agg(
            F.sum("TotalWTWRevenueUSD").alias("Total"),
            F.avg("TotalWTWRevenueUSD").alias("Avg"),
            F.min("TotalWTWRevenueUSD").alias("Min"),
            F.max("TotalWTWRevenueUSD").alias("Max"),
        )
    )
else:
    print("Skipping data quality overview (RUN_STATS = False).")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 14, Finished, Available, Finished, False)

Skipping data quality overview (RUN_STATS = False).


In [13]:
# Cell 12: Write to Gold Lakehouse

fact_transaction.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "Gold_SalesOps_Fact_Transaction"
)

final_count = spark.read.table("Gold_SalesOps_Fact_Transaction").count()
print(f"Gold_SalesOps_Fact_Transaction written: {final_count:,} rows")
print("=== Notebook 03 complete ===")

StatementMeta(, 7b38340a-57df-4424-b206-5fc2d6352fbf, 15, Finished, Available, Finished, False)

Gold_SalesOps_Fact_Transaction written: 426,082,942 rows
=== Notebook 03 complete ===
